<a href="https://colab.research.google.com/github/jsoook-dt/python-colab-analysis-study/blob/sook/project02_sook_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import kagglehub

# 1. Download the latest version of the dataset
# The original dataset handle 'faviovaz/marketing-ab-testing/dataset' was invalid.
# Updated to the correct handle 'mahmoudshogaa/marketing-campaign-ab-testing'.
path = kagglehub.dataset_download("faviovaz/marketing-ab-testing")
print("Dataset path:", path)

# 2. Load the CSV file into a DataFrame
# Combines the download path with the specific filename
df = pd.read_csv(f"{path}/marketing_AB.csv")

# 3. Verify data loading (Check the first 5 rows)
print("Data Sample:")
display(df.head())

# 4. Check data types and structure (Crucial for marketing analysis)
print("\nData Information:")
df.info()

Using Colab cache for faster access to the 'marketing-ab-testing' dataset.
Dataset path: /kaggle/input/marketing-ab-testing
Data Sample:


,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14



Data Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Unnamed: 0     588101 non-null  int64 
 1   user id        588101 non-null  int64 
 2   test group     588101 non-null  object
 3   converted      588101 non-null  bool  
 4   total ads      588101 non-null  int64 
 5   most ads day   588101 non-null  object
 6   most ads hour  588101 non-null  int64 
dtypes: bool(1), int64(4), object(2)
memory usage: 27.5+ MB


In [ ]:
# 2. 데이터 전처리
# 2-1. 결측치 확인
print("결측치:\n", df.isnull().sum())
print("\n")

# 2-2. 중복값 확인
print("중복값:\n", df.duplicated().sum())
print("\n")

# 2-3. 불필요한 컬럼 제거
df = df.drop(columns=['Unnamed: 0'])

display(df.head())

결측치:
 Unnamed: 0       0
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64


중복값:
 0




,user id,test group,converted,total ads,most ads day,most ads hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14


In [ ]:
# 3. EDA - 탐색적 데이터분석
# 3-1. 그룹분포 (사용자 수, 비율, 전환 수, 전환율)
user_count = df.groupby('test group')['converted'].count()
user_ratio = user_count / len(df)
conversion_count = df.groupby('test group')['converted'].sum()
conversion_rate = conversion_count / user_count

###결과확인
result_df = pd.DataFrame({
    'User Count': user_count,
    'User Ratio': user_ratio,
    'Conversion Count': conversion_count,
    'Conversion Rate': conversion_rate
})
## 결과 확인 (출력 시 % 포맷팅 적용)
print("\n### 3-1. 그룹별 분포 요약 ###")
display(result_df.style.format({
    'User Ratio': '{:.2%}',
    'Conversion Rate': '{:.2%}'
}))


### 3-1. 그룹별 분포 요약 ###


,User Count,User Ratio,Conversion Count,Conversion Rate
test group,,,,
ad,564577,96.00%,14423,2.55%
psa,23524,4.00%,420,1.79%


In [70]:
# 3-2. 요일별(사용자 수, 전환 수, 전환율)
## 요일별 사용자 수
day_user_count = df.groupby('most ads day')['converted'].count()
## 요일별 전환 수
day_conversion_count = df.groupby('most ads day')['converted'].sum()
## 요일별 전환비율
day_conversion_rate = day_conversion_count / day_user_count

###결과확인
result_df = pd.DataFrame({
    'User Count': day_user_count,
    'Conversion Count': day_conversion_count,
    'Conversion Rate': day_conversion_rate
})

print("\n### 3-2. 요일별 결과###")
display(result_df.style.format({
    'Conversion Rate': '{:.2%}'
}))



### 3-2. 요일별 결과###


,User Count,Conversion Count,Conversion Rate
most ads day,,,
Friday,92608,2057,2.22%
Monday,87073,2857,3.28%
Saturday,81660,1719,2.11%
Sunday,85391,2090,2.45%
Thursday,82982,1790,2.16%
Tuesday,77479,2312,2.98%
Wednesday,80908,2018,2.49%


In [73]:
# 전환율 높은 순으로
day_stats = df.groupby('most ads day')['converted'].agg(['count', 'sum', 'mean'])
day_stats.columns = ['User Count', 'Conversion Count', 'Conversion Rate']
display(day_stats.sort_values(by='Conversion Rate', ascending=False).style.format({
    'Conversion Rate': '{:.2%}'
}))

,User Count,Conversion Count,Conversion Rate
most ads day,,,
Monday,87073,2857,3.28%
Tuesday,77479,2312,2.98%
Wednesday,80908,2018,2.49%
Sunday,85391,2090,2.45%
Friday,92608,2057,2.22%
Thursday,82982,1790,2.16%
Saturday,81660,1719,2.11%


In [63]:
# 3-3. 시간대별 패턴 (사용자 수, 전환 수, 전환율)
hour_stats = df.groupby('most ads hour')['converted'].agg(['count', 'sum', 'mean'])
hour_stats.columns = ['User Count', 'Conversion Count', 'Conversion Rate']


## 결과 확인 (전환율 기준 내림차순 상위 5개)
print("\n### 3-3-2. 시간대별 분석 (전환율 상위 5개) ###")
display(hour_stats.sort_values(by='Conversion Rate', ascending=False).head(5).style.format({
    'Conversion Rate': '{:.2%}'
}))


### 3-3-2. 시간대별 분석 (전환율 상위 5개) ###


,User Count,Conversion Count,Conversion Rate
most ads hour,,,
16,37567,1156,3.08%
20,28923,862,2.98%
15,44683,1325,2.97%
21,29976,867,2.89%
17,34988,987,2.82%


In [87]:
print("\n### 3-4. 광고 노출 분포 요약 ###")
print(f"평균 노출 수: {ads_mean:.2f}회")
print(f"중앙값 노출 수: {ads_median:.2f}회")
print(f"최대 노출 수: {ads_max}회")
print()

# 노출 횟수를 10개 구간으로 나누어 전환율 확인
df['ads_bin'] = pd.qcut(df['total ads'], q=10, duplicates='drop')
print(df.groupby('ads_bin')['converted'].mean())


### 3-4. 광고 노출 분포 요약 ###
평균 노출 수: 24.82회
중앙값 노출 수: 13.00회
최대 노출 수: 2065회

ads_bin
(0.999, 2.0]      0.001898
(2.0, 3.0]        0.002826
(3.0, 5.0]        0.003509
(5.0, 8.0]        0.004271
(8.0, 13.0]       0.006827
(13.0, 17.0]      0.007797
(17.0, 24.0]      0.012985
(24.0, 33.0]      0.022630
(33.0, 57.0]      0.051726
(57.0, 2065.0]    0.143387
Name: converted, dtype: float64


/tmp/ipykernel_11107/829840656.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('ads_bin')['converted'].mean())


In [85]:
# 4. 전환율 분석


# 4-1. 요일별 전환율
day_conversion = df.groupby('most ads day')['converted'].mean().sort_values(ascending=False)
print("\n[요일별 전환율]")
display(day_conversion.to_frame().style.format('{:.2%}'))

# 4-2. 시간대 TOP 3
hour_conversion = df.groupby('most ads hour')['converted'].mean().sort_values(ascending=False).head(3)
print("\n[시간대 TOP 3]")
display(hour_conversion.to_frame().style.format('{:.2%}'))

# 4-3. 전체 전환율
total_conversion = df['converted'].mean()
print("\n[전체 전환율]")
print(f"{total_conversion:.2%}")

# 4-4. 그룹별 전환율 (핵심)
group_stats = df.groupby('test group')['converted'].agg(['count', 'sum', 'mean'])
group_stats.columns = ['user_count', 'conversion_count', 'conversion_rate']

print("\n[그룹별 전환율]")
display(group_stats[['conversion_rate']].style.format('{:.2%}'))

# 4-5. 차이 & Lift
conv_ad = group_stats.loc['ad', 'conversion_rate']
conv_psa = group_stats.loc['psa', 'conversion_rate']

diff = conv_ad - conv_psa
lift = diff / conv_psa

print("\n[성과 비교]")
print(f"Ad: {conv_ad:.2%}")
print(f"PSA: {conv_psa:.2%}")
print(f"차이: {diff:.4f} ({diff:.2%}p)")
print(f"Lift: {lift:.2%}")


[요일별 전환율]


,converted
most ads day,
Monday,3.28%
Tuesday,2.98%
Wednesday,2.49%
Sunday,2.45%
Friday,2.22%
Thursday,2.16%
Saturday,2.11%



[시간대 TOP 3]


,converted
most ads hour,
16,3.08%
20,2.98%
15,2.97%



[전체 전환율]
2.52%

[그룹별 전환율]


,conversion_rate
test group,
ad,2.55%
psa,1.79%



[성과 비교]
Ad: 2.55%
PSA: 1.79%
차이: 0.0077 (0.77%p)
Lift: 43.09%


In [86]:
# 5. 통계 가설점검
## 귀무가설 (H₀): Ad 그룹과 PSA 그룹의 전환율 차이 없음
## 대립가설 (H₁): Ad 그룹 전환율 > PSA 그룹 전환율

#
from statsmodels.stats.proportion import proportions_ztest

# 5. 통계 가설 검정 (Two-Proportion Z-Test)
print("\n[5.통계 가설 검정]")

# 데이터 준비
counts = df.groupby('test group')['converted'].sum()  # 전환 수 [ad, psa]
nobs = df.groupby('test group')['converted'].count() # 전체 사용자 수 [ad, psa]

# Z-test 실시 (alternative='larger'는 ad > psa 인지 검정)
z_stat, p_val = proportions_ztest(counts, nobs, alternative='larger')

print(f"Z-통계량: {z_stat:.4f}")
print(f"P-value: {p_val:.10f}")

if p_val < 0.05:
    print("결과: ✅ 귀무가설 기각 (Ad 그룹의 전환율이 통계적으로 유의미하게 높음)")
else:
    print("결과: 귀무가설 채택 (두 그룹 간 유의미한 차이가 없음)")


[5.통계 가설 검정]
Z-통계량: 7.3701
P-value: 0.0000000000
결과: ✅ 귀무가설 기각 (Ad 그룹의 전환율이 통계적으로 유의미하게 높음)
